# P1-E05. CAUEEG Channel, Signal-Type and Montage Harmonisation Lock

**Paper 1, Experiment 05 — uniquely versioned successor to the previously strict-photic P1-E04. notebook**

## Goal

Create a diagnosis-blinded, recording-level channel lock for the 979 P1-E04-eligible CAUEEG recordings. This notebook reads EDF headers only, normalises channel labels, separates EEG from auxiliary EKG/Photic signals, verifies a common 19-channel EEG layout, preserves the official train/validation/test assignments, and writes the only channel-harmonised manifests permitted for preprocessing.

Run all cells in order. Continue only when the final status is:

> **P1-E05. PASSED — CHANNEL AND MONTAGE MANIFEST LOCKED**

## Context & Methods

### Key assumptions and prespecified rules

1. The only inputs are P1-E04's locked interval CSV, locked eligible-recording CSV, and its run record.
2. Both input SHA-256 values must reproduce the hashes in the P1-E04 run record.
3. `recording_id` is the analysis unit; released serial numbers are not inferred to be participant IDs.
4. EDF headers are inspected without loading waveform samples.
5. Channel harmonisation is diagnosis-blinded: label and split columns are unavailable while channel rules are derived and applied.
6. The canonical CAUEEG EEG layout is fixed prospectively as 19 10–20 channels: Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2.
7. Common aliases T7/T8/P7/P8 map to T3/T4/T5/T6. Prefixes, suffixes and reference decorations are removed only for name matching; no rereferencing occurs here.
8. EKG/ECG and Photic/Stim channels are catalogued but excluded from the EEG analysis layout.
9. A recording is eligible only if all 19 canonical EEG channels occur exactly once and no unexpected unresolved signal remains.
10. Channel order differences are corrected by an explicit stored index map; data are not yet filtered, rereferenced, resampled, repaired, interpolated or otherwise preprocessed.
11. Official splits are carried forward unchanged. The test split does not influence the rules.

## Setup

### 1. Paths, dependencies and output contract

In [ ]:
from pathlib import Path
import json, re, sys, platform, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

try:
    import mne
except ImportError as exc:
    raise ImportError("Install MNE in this kernel first: pip install mne") from exc

PROJECT_ROOT = Path(r'D:\EEG_AD_Research')
PAPER1_ROOT = PROJECT_ROOT/'paper1_experiments'
E04_ROOT = PAPER1_ROOT/'paper1_outputs'/'p1_e04_event_interval_audit'
INPUT_INTERVALS = E04_ROOT/'tables'/'10_LOCKED_NONPHOTIC_EYES_CLOSED_INTERVALS.csv'
INPUT_ELIGIBLE = E04_ROOT/'tables'/'17_LOCKED_PREPROCESSING_ELIGIBLE_RECORDING_MANIFEST.csv'
E04_RUN_RECORD = E04_ROOT/'run_record.json'

# The versioned directory prevents collision with any earlier P1-E05 output.
OUTPUT_ROOT = PAPER1_ROOT/'paper1_outputs'/'p1_e05_1_channel_montage_lock'
TABLE_DIR, FIGURE_DIR = OUTPUT_ROOT/'tables', OUTPUT_ROOT/'figures'
for directory in (OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ['Normal','MCI','AD']
SPLIT_ORDER = ['train','validation','test']
CANONICAL_EEG_CHANNELS = [
    'Fp1','Fp2','F7','F3','Fz','F4','F8','T3','C3','Cz',
    'C4','T4','T5','P3','Pz','P4','T6','O1','O2'
]
EXPECTED_ELIGIBLE_COUNTS = pd.DataFrame(
    {'Normal':[350,31,33], 'MCI':[305,26,33], 'AD':[177,10,14]},
    index=SPLIT_ORDER,
)

mpl.rcParams.update({'font.family':'Arial','font.size':8,'axes.titlesize':9,
    'axes.labelsize':8,'legend.fontsize':7,'pdf.fonttype':42,'ps.fonttype':42,
    'figure.dpi':120,'axes.spines.top':False,'axes.spines.right':False})
sns.set_theme(style='ticks', context='paper')

def sha256_file(path, block_size=4*1024*1024):
    digest=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(block_size),b''):
            digest.update(block)
    return digest.hexdigest()

def save_table(frame,name,index=False):
    frame.to_csv(TABLE_DIR/f'{name}.csv',index=index)
    try:
        frame.to_excel(TABLE_DIR/f'{name}.xlsx',index=index)
    except Exception as exc:
        warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')

def save_figure(fig,name,width_mm=180,height_mm=90):
    fig.set_size_inches(width_mm/25.4,height_mm/25.4)
    fig.tight_layout()
    for ext in ('png','pdf','svg'):
        fig.savefig(FIGURE_DIR/f'{name}.{ext}',dpi=600 if ext=='png' else None,
                    bbox_inches='tight',facecolor='white')

required_inputs = {
    'P1-E04 interval lock': INPUT_INTERVALS,
    'P1-E04 eligible-recording lock': INPUT_ELIGIBLE,
    'P1-E04 run record': E04_RUN_RECORD,
}
missing=[f'{name}: {path}' for name,path in required_inputs.items() if not path.is_file()]
if missing:
    raise FileNotFoundError('Required P1-E04 outputs are missing:\n'+'\n'.join(missing))
print('Versioned experiment: P1-E05.1')
print('Input root:', E04_ROOT)
print('Output root:', OUTPUT_ROOT)

## Data

### 2. Authenticate both P1-E04 locks

In [ ]:
with E04_RUN_RECORD.open('r',encoding='utf-8') as handle:
    e04_run=json.load(handle)

expected_interval_hash=e04_run.get('locked_interval_sha256','')
expected_eligible_hash=e04_run.get('preprocessing_eligible_manifest_sha256','')
observed_interval_hash=sha256_file(INPUT_INTERVALS)
observed_eligible_hash=sha256_file(INPUT_ELIGIBLE)

eligible=pd.read_csv(INPUT_ELIGIBLE,dtype={'recording_id':str,'serial':str,'identity_group':str})
intervals=pd.read_csv(INPUT_INTERVALS,dtype={'recording_id':str,'serial':str})
for frame in (eligible,intervals):
    for col in ('recording_id','serial'):
        frame[col]=frame[col].astype(str).str.strip().str.zfill(5)

required_eligible={'recording_id','serial','official_split','experimental_label','edf_path',
                   'analysis_unit','participant_identity_verified','retained_intervals','retained_seconds'}
required_intervals={'recording_id','serial','interval_number','start_seconds','stop_seconds','duration_seconds'}
if required_eligible-set(eligible.columns):
    raise ValueError(f'Eligible manifest missing: {sorted(required_eligible-set(eligible.columns))}')
if required_intervals-set(intervals.columns):
    raise ValueError(f'Interval manifest missing: {sorted(required_intervals-set(intervals.columns))}')

eligible['official_split']=eligible.official_split.astype(str).str.strip().str.lower()
eligible['experimental_label']=eligible.experimental_label.astype(str).str.strip()
observed_counts=(eligible.groupby(['official_split','experimental_label']).size()
    .unstack(fill_value=0).reindex(index=SPLIT_ORDER,columns=CLASS_ORDER,fill_value=0).astype(int))
interval_recordings=set(intervals.recording_id)
eligible_recordings=set(eligible.recording_id)
input_checks=pd.DataFrame([
 {'check':'P1-E04 status passed','pass':str(e04_run.get('status','')).startswith('P1-E04 PASSED'),'detail':e04_run.get('status','')},
 {'check':'interval SHA-256 authenticated','pass':bool(expected_interval_hash) and observed_interval_hash==expected_interval_hash,'detail':observed_interval_hash},
 {'check':'eligible-manifest SHA-256 authenticated','pass':bool(expected_eligible_hash) and observed_eligible_hash==expected_eligible_hash,'detail':observed_eligible_hash},
 {'check':'exactly 979 eligible recordings','pass':len(eligible)==979 and eligible.recording_id.nunique()==979,'detail':len(eligible)},
 {'check':'exactly 8,557 locked intervals','pass':len(intervals)==8557,'detail':len(intervals)},
 {'check':'interval and eligible recording sets agree','pass':interval_recordings==eligible_recordings,'detail':{'interval':len(interval_recordings),'eligible':len(eligible_recordings)}},
 {'check':'class-by-split counts reproduce P1-E04','pass':observed_counts.equals(EXPECTED_ELIGIBLE_COUNTS),'detail':observed_counts.to_dict()},
 {'check':'analysis unit remains recording','pass':eligible.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':eligible.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~eligible.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':eligible.participant_identity_verified.value_counts(dropna=False).to_dict()},
 {'check':'all EDF paths exist','pass':eligible.edf_path.map(lambda p:Path(p).is_file()).all(),'detail':int((~eligible.edf_path.map(lambda p:Path(p).is_file())).sum())},
])
save_table(input_checks,'01_p1_e04_input_authentication_checks')
display(observed_counts); display(input_checks)
if not input_checks['pass'].all():
    raise AssertionError('STOP — P1-E04 INPUT AUTHENTICATION FAILED')
print('P1-E04 INPUTS AUTHENTICATED')

## Results

### 3. Read EDF headers diagnosis-blind

In [ ]:
# Labels and split assignments are intentionally absent from this working table.
blind_source=eligible[['recording_id','serial','edf_path']].copy()
header_rows=[]; channel_rows=[]
for row in blind_source.itertuples(index=False):
    try:
        raw=mne.io.read_raw_edf(row.edf_path,preload=False,verbose='ERROR')
        sfreq=float(raw.info['sfreq'])
        n_times=int(raw.n_times)
        names=list(raw.ch_names)
        types=list(raw.get_channel_types())
        header_rows.append({'recording_id':row.recording_id,'serial':row.serial,
            'readable':True,'n_signals':len(names),'sfreq_Hz':sfreq,'n_samples':n_times,
            'duration_seconds':n_times/sfreq,'error':''})
        for index,(name,mne_type) in enumerate(zip(names,types)):
            channel_rows.append({'recording_id':row.recording_id,'serial':row.serial,
                'original_index':index,'original_name':name,'mne_type':mne_type})
        raw.close()
    except Exception as exc:
        header_rows.append({'recording_id':row.recording_id,'serial':row.serial,
            'readable':False,'n_signals':np.nan,'sfreq_Hz':np.nan,'n_samples':np.nan,
            'duration_seconds':np.nan,'error':repr(exc)})

headers=pd.DataFrame(header_rows)
raw_channels=pd.DataFrame(channel_rows)
save_table(headers,'02_edf_header_read_audit')
save_table(raw_channels,'03_original_channel_inventory')
display(headers[['n_signals','sfreq_Hz','duration_seconds']].describe())
display(raw_channels.original_name.value_counts().rename_axis('original_name').reset_index(name='occurrences'))
if not headers.readable.all():
    raise AssertionError('STOP — one or more eligible EDF headers could not be read.')

### 4. Normalise names and classify signal types

Normalisation is conservative and deterministic. It removes common EEG prefixes, reference suffixes and punctuation for matching only. The original EDF labels and indices remain preserved in the audit tables.

In [ ]:
CANONICAL_KEY={re.sub(r'[^A-Z0-9]','',name.upper()):name for name in CANONICAL_EEG_CHANNELS}
ALIAS_KEY={'T7':'T3','T8':'T4','P7':'T5','P8':'T6'}

def compact_channel_name(name):
    text=str(name).strip().upper()
    text=re.sub(r'^(EEG|POLY|CHANNEL|CHAN)\s*[-_:]*\s*','',text)
    # Remove common reference decorations only at the end of a channel label.
    text=re.sub(r'([\s_-]*(REF|LE|RE|AVG|AVERAGE|A1|A2|M1|M2))+$','',text)
    return re.sub(r'[^A-Z0-9]','',text)

def classify_channel(name):
    key=compact_channel_name(name)
    if re.search(r'(PHOTIC|PHOTO|FLASH|STIM|TRIGGER)',key):
        return 'photic',None,key
    if re.search(r'(ECG|EKG|HEART|CARD)',key):
        return 'ekg',None,key
    eeg_key=ALIAS_KEY.get(key,key)
    if eeg_key in CANONICAL_KEY:
        return 'eeg',CANONICAL_KEY[eeg_key],key
    return 'unresolved',None,key

classified=raw_channels.copy()
classified[['signal_type','canonical_name','normalized_key']]=classified.original_name.apply(
    lambda x:pd.Series(classify_channel(x)))

catalog=(classified.groupby(['original_name','normalized_key','signal_type','canonical_name'],dropna=False)
    .agg(occurrences=('recording_id','size'),recordings=('recording_id','nunique'))
    .reset_index().sort_values(['signal_type','canonical_name','original_name'],na_position='last'))
save_table(classified,'04_channel_classification_by_recording')
save_table(catalog,'05_channel_name_and_signal_type_catalog')
display(catalog)

# These are descriptive, diagnosis-blind header observations—not model inputs.
recording_layout=(classified.groupby('recording_id').agg(
    total_signals=('original_name','size'),
    eeg_signals=('signal_type',lambda x:(x=='eeg').sum()),
    ekg_signals=('signal_type',lambda x:(x=='ekg').sum()),
    photic_signals=('signal_type',lambda x:(x=='photic').sum()),
    unresolved_signals=('signal_type',lambda x:(x=='unresolved').sum()),
    unique_canonical_eeg=('canonical_name',lambda x:x.dropna().nunique()),
).reset_index())

canonical_counts=(classified.loc[classified.signal_type.eq('eeg')]
    .groupby(['recording_id','canonical_name']).size().rename('occurrences').reset_index())
duplicates=(canonical_counts.loc[canonical_counts.occurrences.ne(1)].copy())
present=(canonical_counts.pivot(index='recording_id',columns='canonical_name',values='occurrences')
         .fillna(0).reindex(index=blind_source.recording_id,columns=CANONICAL_EEG_CHANNELS,fill_value=0))
missing_long=(present.eq(0).stack().loc[lambda x:x].rename('missing').reset_index()
              .drop(columns='missing'))
save_table(recording_layout,'06_recording_signal_type_counts')
save_table(duplicates,'07_duplicate_canonical_eeg_channels')
save_table(missing_long,'08_missing_canonical_eeg_channels')
display(recording_layout.describe())
print('Missing canonical channel instances:',len(missing_long))
print('Duplicate canonical channel instances:',len(duplicates))

### 5. Freeze the canonical order and channel-selection map

In [ ]:
# One input index for every canonical channel, in the fixed canonical order.
eeg_rows=classified.loc[classified.signal_type.eq('eeg')].copy()
eeg_rows['canonical_index']=eeg_rows.canonical_name.map({c:i for i,c in enumerate(CANONICAL_EEG_CHANNELS)})
channel_map=(eeg_rows.sort_values(['recording_id','canonical_index','original_index'])
    [['recording_id','serial','original_index','original_name','normalized_key',
      'canonical_index','canonical_name','mne_type']].reset_index(drop=True))

duplicate_recordings=set(duplicates.recording_id)
missing_recordings=set(missing_long.recording_id)
unresolved_recordings=set(classified.loc[classified.signal_type.eq('unresolved'),'recording_id'])
layout_lock=recording_layout.copy()
layout_lock['has_duplicate_canonical_eeg']=layout_lock.recording_id.isin(duplicate_recordings)
layout_lock['has_missing_canonical_eeg']=layout_lock.recording_id.isin(missing_recordings)
layout_lock['has_unresolved_signal']=layout_lock.recording_id.isin(unresolved_recordings)
layout_lock['channel_layout_eligible']=(
    layout_lock.eeg_signals.eq(len(CANONICAL_EEG_CHANNELS)) &
    layout_lock.unique_canonical_eeg.eq(len(CANONICAL_EEG_CHANNELS)) &
    ~layout_lock.has_duplicate_canonical_eeg &
    ~layout_lock.has_missing_canonical_eeg &
    ~layout_lock.has_unresolved_signal
)
layout_lock['channel_layout_reason']=np.select([
    layout_lock.has_unresolved_signal,
    layout_lock.has_duplicate_canonical_eeg,
    layout_lock.has_missing_canonical_eeg,
],['unresolved_signal','duplicate_canonical_eeg','missing_canonical_eeg'],default='eligible')
layout_lock['canonical_layout']='|'.join(CANONICAL_EEG_CHANNELS)
layout_lock['channel_rule_version']='P1-E05.1-v1'

# Join diagnosis and split only after the diagnosis-blind channel lock is frozen.
audit_columns=['recording_id','serial','official_split','experimental_label','experimental_label_code',
               'age','analysis_unit','identity_group','participant_identity_verified','split_source',
               'edf_path','event_path','duration_seconds','sfreq_Hz','n_samples','edf_sha256',
               'event_sha256','manifest_row_sha256','retained_intervals','retained_seconds',
               'retained_interval_count','retained_seconds_verified','interval_manifest_version']
audit_columns=[c for c in audit_columns if c in eligible.columns]
# `recording_layout` is grouped by recording_id, so `serial` is intentionally
# restored from the authenticated P1-E04 manifest during this one-to-one join.
layout_with_labels=layout_lock.merge(eligible[audit_columns],on='recording_id',
                                     how='left',validate='one_to_one')
final_recordings=layout_with_labels.loc[layout_with_labels.channel_layout_eligible].copy()
final_intervals=intervals.merge(
    final_recordings[['recording_id','channel_rule_version','canonical_layout']],
    on='recording_id',how='inner',validate='many_to_one')
final_channel_map=channel_map.loc[channel_map.recording_id.isin(final_recordings.recording_id)].copy()

save_table(channel_map,'09_diagnosis_blinded_channel_selection_map')
save_table(layout_lock,'10_diagnosis_blinded_recording_channel_layout_lock')
save_table(layout_with_labels,'11_channel_layout_audit_with_labels')
save_table(final_channel_map,'12_LOCKED_CANONICAL_19_EEG_CHANNEL_SELECTION_MAP')
save_table(final_recordings,'13_LOCKED_CHANNEL_HARMONISED_RECORDING_MANIFEST')
save_table(final_intervals,'14_LOCKED_CHANNEL_HARMONISED_INTERVAL_MANIFEST')

summary=(layout_with_labels.groupby(['official_split','experimental_label']).agg(
    p1_e04_eligible=('recording_id','size'),
    channel_harmonised=('channel_layout_eligible','sum')).reset_index())
summary['retention_fraction']=summary.channel_harmonised/summary.p1_e04_eligible
save_table(summary,'15_channel_harmonisation_summary_by_split_and_class')
display(summary)
display(layout_lock.channel_layout_reason.value_counts().rename_axis('reason').reset_index(name='recordings'))

### 6. Publication figures

In [ ]:
signal_counts=(classified.signal_type.value_counts().reindex(
    ['eeg','ekg','photic','unresolved'],fill_value=0).rename_axis('signal_type').reset_index(name='channel_instances'))
fig,axes=plt.subplots(1,2)
sns.barplot(data=signal_counts,x='signal_type',y='channel_instances',hue='signal_type',
            palette=['#0072B2','#D55E00','#009E73','#CC79A7'],legend=False,ax=axes[0])
axes[0].set(xlabel='Signal type',ylabel='Header channel instances',title='Diagnosis-blind signal classification')
axes[0].tick_params(axis='x',rotation=20)

coverage=(present.gt(0).mean().reindex(CANONICAL_EEG_CHANNELS)*100)
axes[1].bar(coverage.index,coverage.values,color='#56B4E9')
axes[1].axhline(100,color='#333333',lw=.8,ls='--')
axes[1].set(xlabel='Canonical EEG channel',ylabel='Recording coverage (%)',
            title='Canonical 19-channel coverage',ylim=(0,102))
axes[1].tick_params(axis='x',rotation=45)
save_figure(fig,'figure_1_signal_types_and_canonical_channel_coverage',180,82); plt.show()

retention=(summary.groupby('experimental_label')[['p1_e04_eligible','channel_harmonised']]
           .sum().reindex(CLASS_ORDER))
fig,ax=plt.subplots()
x=np.arange(len(CLASS_ORDER))
ax.bar(x-.18,retention.p1_e04_eligible,.36,label='P1-E04 eligible',color='#BDBDBD')
ax.bar(x+.18,retention.channel_harmonised,.36,label='P1-E05.1 harmonised',color='#0072B2')
ax.set(xticks=x,xticklabels=CLASS_ORDER,ylabel='Recordings',title='Channel-harmonisation retention')
ax.legend(frameon=False)
for container in ax.containers: ax.bar_label(container,padding=2,fontsize=7)
save_figure(fig,'figure_2_channel_harmonisation_retention',120,78); plt.show()

## Checks

### 7. Final integrity gates and reproducibility record

In [ ]:
final_counts=(final_recordings.groupby(['official_split','experimental_label']).size()
    .unstack(fill_value=0).reindex(index=SPLIT_ORDER,columns=CLASS_ORDER,fill_value=0).astype(int))
map_counts=final_channel_map.groupby('recording_id').size() if not final_channel_map.empty else pd.Series(dtype=int)
map_names=(final_channel_map.groupby('recording_id').canonical_name.apply(list)
           if not final_channel_map.empty else pd.Series(dtype=object))
original_split=eligible.set_index('recording_id').official_split
locked_split=final_recordings.set_index('recording_id').official_split

checks=pd.DataFrame([
 {'check':'both P1-E04 sources authenticated','pass':observed_interval_hash==expected_interval_hash and observed_eligible_hash==expected_eligible_hash,'detail':'SHA-256 matched'},
 {'check':'all 979 eligible EDF headers read','pass':headers.readable.all() and len(headers)==979,'detail':f'{int(headers.readable.sum())}/{len(headers)}'},
 {'check':'one header audit row per recording','pass':headers.recording_id.nunique()==979 and not headers.recording_id.duplicated().any(),'detail':headers.recording_id.nunique()},
 {'check':'all signals classified','pass':not classified.empty and classified.signal_type.notna().all(),'detail':classified.signal_type.value_counts().to_dict()},
 {'check':'no unresolved signals','pass':not classified.signal_type.eq('unresolved').any(),'detail':int(classified.signal_type.eq('unresolved').sum())},
 {'check':'no duplicate canonical EEG channels','pass':duplicates.empty,'detail':len(duplicates)},
 {'check':'no missing canonical EEG channels','pass':missing_long.empty,'detail':len(missing_long)},
 {'check':'every locked recording has exactly 19 map rows','pass':len(map_counts)==len(final_recordings) and map_counts.eq(19).all(),'detail':map_counts.value_counts().to_dict()},
 {'check':'every map reproduces fixed canonical order','pass':len(map_names)==len(final_recordings) and map_names.map(lambda x:x==CANONICAL_EEG_CHANNELS).all(),'detail':'Fp1 through O2 canonical order'},
 {'check':'all P1-E04 recordings retained after channel gate','pass':len(final_recordings)==979,'detail':len(final_recordings)},
 {'check':'all 8,557 intervals retained after channel gate','pass':len(final_intervals)==8557,'detail':len(final_intervals)},
 {'check':'class-by-split counts unchanged','pass':final_counts.equals(EXPECTED_ELIGIBLE_COUNTS),'detail':final_counts.to_dict()},
 {'check':'official split assignments unchanged','pass':locked_split.eq(original_split.loc[locked_split.index]).all(),'detail':final_recordings.official_split.value_counts().to_dict()},
 {'check':'all endpoint classes remain in every split','pass':final_counts.gt(0).all().all(),'detail':final_counts.to_dict()},
 {'check':'analysis remains recording-level','pass':final_recordings.analysis_unit.astype(str).str.lower().eq('recording').all(),'detail':final_recordings.analysis_unit.value_counts().to_dict()},
 {'check':'participant identity remains unverified','pass':~final_recordings.participant_identity_verified.astype(str).str.lower().isin(['true','1']).any(),'detail':final_recordings.participant_identity_verified.value_counts(dropna=False).to_dict()},
 {'check':'channel construction remained diagnosis-blind','pass':'experimental_label' not in blind_source.columns and 'experimental_label' not in classified.columns and 'experimental_label' not in layout_lock.columns,'detail':'labels joined after layout lock'},
 {'check':'no signal preprocessing performed','pass':True,'detail':'EDF headers only; no waveform samples loaded'},
])
save_table(checks,'16_final_integrity_checks')
display(checks)

status=('P1-E05.1 PASSED — CHANNEL AND MONTAGE MANIFEST LOCKED'
        if checks['pass'].all() else 'STOP — P1-E05.1 CHECKS FAILED; DO NOT BEGIN PREPROCESSING')

locked_map_csv=TABLE_DIR/'12_LOCKED_CANONICAL_19_EEG_CHANNEL_SELECTION_MAP.csv'
locked_recordings_csv=TABLE_DIR/'13_LOCKED_CHANNEL_HARMONISED_RECORDING_MANIFEST.csv'
locked_intervals_csv=TABLE_DIR/'14_LOCKED_CHANNEL_HARMONISED_INTERVAL_MANIFEST.csv'
run_record={
 'experiment':'P1-E05.1 CAUEEG channel, signal-type and montage harmonisation lock',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),
 'source_interval_csv':str(INPUT_INTERVALS),'source_interval_sha256':observed_interval_hash,
 'source_eligible_csv':str(INPUT_ELIGIBLE),'source_eligible_sha256':observed_eligible_hash,
 'input_recordings':len(eligible),'input_intervals':len(intervals),
 'channel_harmonised_recordings':len(final_recordings),'channel_harmonised_intervals':len(final_intervals),
 'canonical_eeg_channels':CANONICAL_EEG_CHANNELS,
 'canonical_channel_count':len(CANONICAL_EEG_CHANNELS),
 'signal_type_counts':{str(k):int(v) for k,v in classified.signal_type.value_counts().items()},
 'locked_channel_map_csv':str(locked_map_csv),'locked_channel_map_sha256':sha256_file(locked_map_csv),
 'locked_recording_manifest_csv':str(locked_recordings_csv),'locked_recording_manifest_sha256':sha256_file(locked_recordings_csv),
 'locked_interval_manifest_csv':str(locked_intervals_csv),'locked_interval_manifest_sha256':sha256_file(locked_intervals_csv),
 'identity_interpretation':{'analysis_unit':'recording','serial_to_patient_mapping_available':False,
   'held_out_protection':'official CAUEEG no-overlap split carried forward unchanged'},
 'operations_explicitly_not_performed':['waveform loading','filtering','notch filtering','resampling',
    'rereferencing','artifact correction','bad-channel interpolation','feature extraction','modeling'],
 'mne':mne.__version__,'python':sys.version,'platform':platform.platform(),'status':status,
}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not checks['pass'].all():
    raise AssertionError(status)

## Next Steps

After a complete PASS:

1. Treat `12_LOCKED_CANONICAL_19_EEG_CHANNEL_SELECTION_MAP.csv`, `13_LOCKED_CHANNEL_HARMONISED_RECORDING_MANIFEST.csv`, and `14_LOCKED_CHANNEL_HARMONISED_INTERVAL_MANIFEST.csv` as the only permitted inputs to P1-E06.
2. P1-E06 should perform the prespecified preprocessing-branch implementation and diagnosis-blinded signal-quality audit.
3. Do not infer that a recording serial number is a participant identifier.
4. If an external dataset later supplies verified participant IDs, preserve this same channel pipeline but replace recording-level grouping with participant-level grouping before any split or cross-validation operation.
5. Never use the test split to select a montage, reference, preprocessing branch, quality threshold, representation, model or hyperparameter.